# 02 — PaDEL Fingerprint Matrix Construction

> Clean senior-review version. This notebook contains only the final, logically connected pipeline. 
> Exploratory/probing/failed scripts are intentionally excluded from the main workflow.

## Objective
Calculate PaDEL molecular fingerprints from the curated RecA dataset and export one modeling-ready matrix.

In [ ]:
# !apt-get install -y default-jre
# !pip install -q pandas numpy padelpy

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()
CURATED_FILE = PROJECT_DIR / "outputs" / "recA_chembl" / "curated" / "recA_curated_ec50_binary.csv"
FP_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "fingerprints"
FP_DIR.mkdir(parents=True, exist_ok=True)

SMILES_FILE = FP_DIR / "recA_padel_input.smi"
FINGERPRINT_FILE = FP_DIR / "recA_padel_fingerprints.csv"
MODELING_MATRIX_FILE = FP_DIR / "recA_modeling_matrix.csv"

## 1. Prepare SMILES input

In [ ]:
df = pd.read_csv(CURATED_FILE)
required = ["Name", "SMILES", "bioactivity_class"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

smiles_df = df[["SMILES", "Name"]].dropna().drop_duplicates("Name")
smiles_df.to_csv(SMILES_FILE, sep="\t", header=False, index=False)

print(f"Saved PaDEL input: {SMILES_FILE}")
smiles_df.head()

## 2. Calculate PaDEL fingerprints

In [ ]:
from padelpy import padeldescriptor

padeldescriptor(
    mol_dir=str(SMILES_FILE),
    d_file=str(FINGERPRINT_FILE),
    fingerprints=True,
    descriptors=False,
    detectaromaticity=True,
    standardizenitro=True,
    standardizetautomers=True,
    threads=2,
    removesalt=True,
    log=True,
)

print(f"Saved fingerprints: {FINGERPRINT_FILE}")

## 3. Assemble modeling matrix

In [ ]:
fp = pd.read_csv(FINGERPRINT_FILE)

# PaDEL commonly exports molecule name as Name
if "Name" not in fp.columns:
    possible_name_cols = [c for c in fp.columns if c.lower() in ["name", "molecule", "compound"]]
    if possible_name_cols:
        fp = fp.rename(columns={possible_name_cols[0]: "Name"})
    else:
        raise ValueError("Cannot identify compound Name column in PaDEL output.")

meta = df[["Name", "SMILES", "EC50_nM", "pEC50", "bioactivity_class"]].copy()
matrix = meta.merge(fp, on="Name", how="inner")

# Clean fingerprint columns
feature_cols = [c for c in matrix.columns if c not in ["Name", "SMILES", "EC50_nM", "pEC50", "bioactivity_class"]]
matrix[feature_cols] = matrix[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

matrix.to_csv(MODELING_MATRIX_FILE, index=False)

print(matrix.shape)
print(f"Saved modeling matrix: {MODELING_MATRIX_FILE}")
matrix.head()

## Final output
`outputs/recA_chembl/fingerprints/recA_modeling_matrix.csv`